# Week X – Batch Processing with Apache Spark

StableFlow issuer-event data extracted from Postgres into chunked Parquet files.

**Dataset:** `/home/freeman/stablecoin-flow/data/raw_issuer_events/`

| Property | Value |
|----------|-------|
| Files | 32 Parquet files (`raw_issuer_events_00000.parquet` … `00031.parquet`) |
| Total rows | 991,540 |
| Total size | ~79 MB |
| Date range | 2026-05-01 → 2026-11-06 |
| Tokens | USDC, USDT |
| Event types | MINT, BURN, BLACKLIST, UNBLACKLIST, DESTROY_BLACKLIST |

## Schema (flat, one table — no normal forms)

| Column | Type | Notes |
|--------|------|-------|
| `block_number` | long | Ethereum block number |
| `block_timestamp` | timestamp (UTC, microseconds) | Block time |
| `event_date` | date | Derived from `block_timestamp` |
| `transaction_hash` | string | Lowercase, `0x` prefix |
| `log_index` | int | Log index within the transaction |
| `contract_address` | string | Lowercase, `0x` prefix |
| `token` | string | `USDC` / `USDT` |
| `event_type` | string | `MINT`, `BURN`, `BLACKLIST`, `UNBLACKLIST`, `DESTROY_BLACKLIST` |
| `topic1_address` | string | Decoded from padded topic, lowercase `0x` |
| `topic2_address` | string | Decoded from padded topic, lowercase `0x` |
| `amount_raw` | string | uint256 from hex `data`, stored as decimal string |
| `blacklisted_address` | string | Address blacklisted/unblacklisted |
| `destroyed_amount_raw` | long | Raw destroyed amount (DESTROY_BLACKLIST only) |
| `destroyed_amount_usdt` | decimal(38,6) | USD value destroyed |
| `ingested_at` | timestamp (UTC) | ETL load timestamp |

Natural key: `(transaction_hash, log_index)`.

## Spark Cluster, Master, and `local[*]`

### What a Spark cluster is

A Spark application runs on a **cluster** made of two kinds of processes:

- **Driver** — the "brain". It holds your Python code, builds the execution plan,
  schedules tasks, and collects results. It runs your `main()` or notebook cells.
- **Executors** — the "muscle". They do the actual work: reading files,
  filtering rows, aggregating, writing output. Each executor has its own JVM and
  memory, and runs one or more **tasks** in parallel.

A **cluster manager** (Standalone, YARN, Kubernetes, etc.) decides which executor
gets which task and what resources they use.

### What `.master(...)` means

`.master()` tells Spark **where to find the cluster manager** — i.e., how to
start the driver and talk to executors. Common values:

| Master value | Meaning |
|--------------|---------|
| `local` | Run everything in one thread on your laptop (no parallelism). |
| `local[*]` | Run on your laptop using **all available CPU cores** as separate threads. |
| `local[4]` | Use exactly 4 threads. |
| `spark://host:7077` | Connect to a real Spark Standalone cluster (driver + remote executors). |
| `yarn` / `k8s` | Connect to a Hadoop YARN or Kubernetes cluster. |

For this lesson we use `local[*]`:

- It runs entirely on your machine — no separate cluster to install.
- `*` means "use every CPU core your computer has." If you have 8 cores, Spark
  starts 8 executor threads, so it can read and process multiple Parquet files at
  once.
- You can still see task parallelism in the Spark UI: each of the 32 Parquet
  files becomes a task, and your cores run several of them simultaneously.

This is the same pattern you saw in the ZoomCamp reference notebooks
(`01_spark_dataFrames.ipynb` … `06_local_cluster.ipynb`), where `06` connects
to `spark://DESKTOP-LQFDL79.localdomain:7077` — a real cluster instead of
`local[*]`. The rest of the code is identical; only the master changes.

## Setup

Create a `SparkSession` pointing at a local cluster. Keep the Spark UI open at `http://localhost:4040` while running — you will see task parallelism across the 32 files in the **Stages** tab.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("stablecoin-issuer-lesson")
    .getOrCreate()
)

print("Spark UI available at http://localhost:4040")
print("Spark version:", spark.version)


ModuleNotFoundError: No module named 'pyspark'

## Step 1 — Read & Inspect

Load the chunked Parquet directory into a single DataFrame, inspect the schema, count rows, and show a sample.

**Spark UI checkpoint:** Open `http://localhost:4040` → **Stages**. You should see ~32 tasks (one per Parquet file) in the read stage — this is task parallelism in action.

In [ ]:
path = "data/raw_issuer_events"

df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(path)
)

print("=== Schema ===")
df.printSchema()

print(f"\nTotal rows: {df.count():,}")

print("\n=== Sample ===")
df.show(5, truncate=False)


## Step 2 — Explore

Look at distinct values, filter by token or date, and count rows per category.

**Spark UI checkpoint:** `distinct()` and `groupBy().count()` trigger shuffle stages; `filter()` is narrow (no shuffle — each task filters its own files locally).

In [ ]:
print("=== Distinct event types ===")
df.select("event_type").distinct().show()

print("=== Distinct tokens ===")
df.select("token").distinct().show()

print("=== MINT events (first 5) ===")
df.filter(F.col("event_type") == "MINT").show(5, truncate=False)

print("=== BURN events (first 5) ===")
df.filter(F.col("event_type") == "BURN").show(5, truncate=False)

print("=== USDC events on 2026-05-07 ===")
df.filter(
    (F.col("token") == "USDC") &
    (F.col("event_date") == "2026-05-07")
).show(5, truncate=False)

print("=== Rows per event type ===")
df.groupBy("event_type").count().orderBy("count", ascending=False).show()

print("=== Rows per token ===")
df.groupBy("token").count().show()


## Step 3 — Transform

Derive new columns, cast types, and add business-logic flags.

**Key concept:** Transformations are lazy — `printSchema()` and `show()` are the actions that finally execute the plan.

In [ ]:
df_t = (
    df
    # amount_raw is stored as string; cast to decimal for math
    .withColumn("amount_usd", F.col("amount_raw").cast(DecimalType(38, 6)))
    # Business flag: is this a mint?
    .withColumn("is_mint", F.col("event_type") == "MINT")
    # Business flag: is this a burn?
    .withColumn("is_burn", F.col("event_type") == "BURN")
    # Derive hour from timestamp for time-series analysis
    .withColumn("event_hour", F.date_trunc("hour", F.col("block_timestamp")))
)

print("=== Transformed sample ===")
df_t.select(
    "event_date", "event_hour", "token", "event_type",
    "amount_usd", "is_mint", "is_burn"
).show(5, truncate=False)

df_t.printSchema()


## Step 4 — Aggregate

Group by business dimensions and compute metrics.

**Spark UI checkpoint:** After `groupBy`, look at the **Stages** tab — you will see a shuffle stage where Spark exchanges data across partitions to aggregate by key.

In [ ]:
print("=== Daily mint/burn volume by token ===")
daily = (
    df_t
    .groupBy("event_date", "token", "event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount_usd").alias("total_amount_usd"),
        F.avg("amount_usd").alias("avg_amount_usd")
    )
    .orderBy("event_date", "token", "event_type")
)

daily.show(20, truncate=False)

print("=== Net flow per token per day (mint - burn) ===")
net = (
    df_t
    .groupBy("event_date", "token")
    .agg(
        F.sum(F.when(F.col("is_mint"), F.col("amount_usd")).otherwise(0)).alias("total_minted"),
        F.sum(F.when(F.col("is_burn"), F.col("amount_usd")).otherwise(0)).alias("total_burned"),
    )
    .withColumn("net_flow", F.col("total_minted") - F.col("total_burned"))
    .orderBy("event_date", "token")
)

net.show(20, truncate=False)


## Step 5 — Spark SQL

Register the DataFrame as a temporary view and run SQL queries. Demo a broadcast join with a small lookup table.

**Key concept:** `broadcast()` sends the small lookup table to all executors, avoiding a shuffle for the join — critical when joining a big fact table with a small dimension table.

In [ ]:
df_t.createOrReplaceTempView("issuer_events")

# Pure SQL aggregation
spark.sql("""
    SELECT
        event_date,
        token,
        event_type,
        COUNT(*) AS event_count,
        SUM(amount_usd) AS total_amount_usd
    FROM issuer_events
    GROUP BY event_date, token, event_type
    ORDER BY event_date DESC, total_amount_usd DESC
    LIMIT 20
""").show(truncate=False)

# Broadcast join with a small lookup table
tokens_lookup = spark.createDataFrame(
    [
        ("USDC", 6, "0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48"),
        ("USDT", 6, "0xdac17f958d2ee523a2206206994597c13d831ec7"),
    ],
    ["token", "decimals", "issuer_address"]
)

df_enriched = (
    df_t
    .join(F.broadcast(tokens_lookup), on="token", how="left")
)

df_enriched.select("token", "decimals", "issuer_address", "event_type", "amount_usd").show(5, truncate=False)


## Step 6 — Write Output

Write the transformed data back to Parquet. Two common patterns:

1. **Partitioned output** (recommended for analytics) — creates subdirectories like `event_date=2026-05-07/token=USDC/`
2. **Single-file output** (good for small reports) via `coalesce(1)`

**Spark UI checkpoint:** After `write`, check the **Stages** tab — you will see a write stage.

In [ ]:
import os

report_dir = "/home/freeman/Curriculum/docs/batch-processing/data/report"
os.makedirs(report_dir, exist_ok=True)

# Option A — Partitioned output (recommended for analytics)
daily = (
    df_t
    .groupBy("event_date", "token", "event_type")
    .agg(
        F.count("*").alias("event_count"),
        F.sum("amount_usd").alias("total_amount_usd"),
    )
)

daily.write.mode("overwrite").partitionBy("event_date", "token").parquet(
    f"{report_dir}/daily_by_token/"
)

# Option B — Single-file output (good for small reports)
net = (
    df_t
    .groupBy("event_date", "token")
    .agg(
        F.sum(F.when(F.col("is_mint"), F.col("amount_usd")).otherwise(0)).alias("total_minted"),
        F.sum(F.when(F.col("is_burn"), F.col("amount_usd")).otherwise(0)).alias("total_burned"),
    )
    .withColumn("net_flow", F.col("total_minted") - F.col("total_burned"))
)

net.coalesce(1).write.mode("overwrite").parquet(
    f"{report_dir}/net_flow_single_file/"
)

print("Written to", report_dir)


## Exercises

**Exercise 1 – prove the read works (5 min).** Run Step 1. Confirm the row count is 991,540 and the schema has 15 columns.

**Exercise 2 – find the biggest day (10 min).** Using Step 4 as a starting point, find which `event_date` has the most total `amount_usd` across all tokens and event types. Print just that row.

**Exercise 3 – broadcast join (10 min).** Extend Step 5 to join against a larger lookup table you create yourself (e.g., 10 rows of fake "exchange" addresses). Compare the Spark UI with and without `broadcast()`.

**Exercise 4 – partitioned read-back (10 min).** After running Step 6 with `partitionBy`, read the output back with a filter:
```python
spark.read.parquet("/home/freeman/Curriculum/docs/batch-processing/data/report/daily_by_token/").filter("event_date = '2026-05-07'").count()
```
Confirm Spark only reads the relevant partition folder(s).

## Check Your Understanding

Answer in your own words:

1. What does `recursiveFileLookup=true` do, and why do we need it here?
2. What is the difference between a narrow transformation (`filter`) and a wide transformation (`groupBy`)? Which one shows shuffle in the Spark UI?
3. Why is `amount_raw` stored as a string instead of a numeric type?
4. When would you use `partitionBy` vs `coalesce(1)` when writing Parquet?
5. What does `broadcast()` do in a join, and when should you use it?

## Assignment

Build a daily net-flow report:

1. Read the full `data/raw_issuer_events/` directory.
2. Cast `amount_raw` to decimal, compute `net_flow = total_minted − total_burned` per `event_date` and `token`.
3. Filter to dates where `|net_flow| > 1,000,000` (large net moves).
4. Write the result as a single-file Parquet using `coalesce(1)`.
5. Verify by reading it back and showing the schema and row count.

## How to Run

Open this notebook in Jupyter and run the cells top to bottom:

```bash
cd /home/freeman/Curriculum/docs/batch-processing
jupyter notebook batch_processing_lesson.ipynb
```

Spark UI will be available at `http://localhost:4040` while a job runs.